# v16 案A: PF全置換 — 学習型再帰フィルタ (GRU + cross-attention)

PFの「GR照合(尤度) + dip動力学(再帰)」をニューラルで再現する直訳版。

- **typewellエンコーダ**: typewell GR を 1D-CNN で符号化 → メモリ(key/value)。value に TVT を付与。
- **水平エンコーダ**: 各点の GR窓 + 幾何(z_rel, md_rel, sbase_rel, known) を符号化 → query。
- **cross-attention**: query が typewell に注意 → 学習版GR尤度のコンテキスト。
- **GRU**: predict-update の再帰を学習 → 残差 r を出力。

ターゲットは `r = s_true - s_base` (s=TVT+Z, s_base=既知dipの線形外挿)。`TVT = s_base + r - Z`。

> 実行はユーザーが行う。CVは GroupKFold(by well) の overall RMSE で v9(6.946 相当)と比較する。

In [1]:
import importlib
import numpy as np
import torch
import torch.nn as nn
import nn_pf_common as C
importlib.reload(C)
print('device:', C.DEVICE)

device: cuda


In [2]:
# 全坑井を前処理（数十MB程度・RAM安全）。残差スケールの妥当性も確認。
wells = C.build_all('train')
r_std, r_mad = C.target_r_std(wells)
print(f'residual r: std={r_std:.2f} ft, mad={r_mad:.2f} ft, TARGET_SCALE={C.TARGET_SCALE}')

  built 100/773
  built 200/773
  built 300/773
  built 400/773
  built 500/773
  built 600/773
  built 700/773
[build_all] 773 wells ready (split=train)
residual r: std=42.40 ft, mad=26.39 ft, TARGET_SCALE=30.0


In [3]:
class ModelA(nn.Module):
    def __init__(self, d=96, gr_ch=32, heads=4, gru_layers=1):
        super().__init__()
        # GR窓エンコーダ（水平・typewell 共有重みでも別でも可。ここでは別々）
        self.h_gr = C.GRConv(gr_ch)
        self.t_gr = C.GRConv(gr_ch)
        # 水平 query: [gr_emb, z_rel, md_rel, sbase_rel, known] -> d
        self.h_proj = nn.Sequential(nn.Linear(gr_ch + 4, d), nn.GELU(), nn.Linear(d, d))
        # typewell key/value: [gr_emb, tvt_norm] -> d
        self.t_proj = nn.Sequential(nn.Linear(gr_ch + 1, d), nn.GELU(), nn.Linear(d, d))
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.gru = nn.GRU(d * 2, d, num_layers=gru_layers, batch_first=True, bidirectional=False)
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))

    def forward(self, b):
        # --- typewell メモリ ---
        t_emb = self.t_gr(b['tw_gr_n'])                       # [B,T,gr_ch]
        vmask = b['tw_valid'].float()                         # [B,T] 1=有効
        tmean = (b['tw_tvt'] * vmask).sum(1, keepdim=True) / vmask.sum(1, keepdim=True).clamp_min(1)
        tvt_n = ((b['tw_tvt'] - tmean) / 1000.0 * vmask).unsqueeze(-1)  # 無効は0（key_padding_maskで無視）
        mem = self.t_proj(torch.cat([t_emb, tvt_n], dim=-1)) # [B,T,d]
        key_pad = ~b['tw_valid']                              # True=無視
        # --- 水平 query ---
        h_emb = self.h_gr(b['gr_n'])                          # [B,H,gr_ch]
        feats = torch.stack([b['z_rel'] / 100.0, b['md_rel'],
                             b['sbase_rel'] / C.TARGET_SCALE, b['known']], dim=-1)
        q = self.h_proj(torch.cat([h_emb, feats], dim=-1))   # [B,H,d]
        ctx, _ = self.attn(q, mem, mem, key_padding_mask=key_pad)  # [B,H,d]
        seq = torch.cat([q, ctx], dim=-1)                    # [B,H,2d]
        out, _ = self.gru(seq)
        r = self.head(out).squeeze(-1)                       # [B,H] = r/TARGET_SCALE
        return r

def make_A():
    return ModelA()

# 形状スモークテスト（1バッチ）
from torch.utils.data import DataLoader
ds = C.WellDataset(wells[:8], mode='train', crop=512)
dl = DataLoader(ds, batch_size=4, collate_fn=C.collate)
b = C.to_device(next(iter(dl)))
m = make_A().to(C.DEVICE)
r = m(b)
print('r shape:', tuple(r.shape), '| pred shape:', tuple(C.tvt_from_r(r, b).shape))

r shape: (4, 512) | pred shape: (4, 512)


In [ ]:
# CV 実行（GroupKFold by well, overall RMSE）。エポック数・バッチは環境に合わせて調整。
cv, oof = C.run_cv(wells, make_A, name='A_gru_attn',
                   n_splits=5, epochs=20, lr=1e-3, batch_size=8, crop=768)
print('CV overall RMSE (A):', cv)

c:\Users\kosaka256\Documents\rogii_claude\v16\nn_pf_common.py:335: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  tot += float(loss); nb += 1


[A_gru_attn] fold0 ep00 train_loss=189.8053 val_overall_rmse=36.3383
[A_gru_attn] fold0 ep01 train_loss=112.8558 val_overall_rmse=22.7729
[A_gru_attn] fold0 ep02 train_loss=95.9168 val_overall_rmse=19.7083
[A_gru_attn] fold0 ep03 train_loss=79.1848 val_overall_rmse=22.0969
[A_gru_attn] fold0 ep04 train_loss=84.7320 val_overall_rmse=18.8018
[A_gru_attn] fold0 ep05 train_loss=84.7022 val_overall_rmse=17.1122
[A_gru_attn] fold0 ep06 train_loss=78.7897 val_overall_rmse=18.1874
[A_gru_attn] fold0 ep07 train_loss=78.6764 val_overall_rmse=20.4928
[A_gru_attn] fold0 ep08 train_loss=80.2663 val_overall_rmse=16.4006
[A_gru_attn] fold0 ep09 train_loss=74.5782 val_overall_rmse=17.0338
[A_gru_attn] fold0 ep10 train_loss=76.4771 val_overall_rmse=15.7518
[A_gru_attn] fold0 ep11 train_loss=75.3159 val_overall_rmse=15.9347
[A_gru_attn] fold0 ep12 train_loss=67.6874 val_overall_rmse=16.0531
[A_gru_attn] fold0 ep13 train_loss=72.0827 val_overall_rmse=15.7573


In [ ]:
# OOF を保存（ブレンド素材・分析用）
import os, pickle
os.makedirs('artifacts_v16', exist_ok=True)
with open('artifacts_v16/oof_A.pkl', 'wb') as f:
    pickle.dump({'cv': cv, 'oof': oof}, f)
print('saved artifacts_v16/oof_A.pkl')